# Notebook Pipeline Map: docs/notebooks/dvc.yaml

This notebook is about the local notebook pipeline file at `docs/notebooks/dvc.yaml`.

For generic DVC concepts and command semantics, use the overview page: [DVC Overview](../overview/dvc).

## What This Pipeline Does

`docs/notebooks/dvc.yaml` defines one stage per notebook, typically named `notebook_<name>`.

Each stage captures:
- `cmd`: how the notebook executes (usually `jupyter nbconvert --execute --inplace`).
- `deps`: notebook and code/config dependencies.
- optional `outs`, `metrics`, `plots`: persisted artifacts for CI and reproducibility.

In [1]:
from pathlib import Path
import yaml

DVC_FILE = Path('dvc.yaml')
payload = yaml.safe_load(DVC_FILE.read_text(encoding='utf-8'))
stages = payload.get('stages', {})

print(f'Stage count: {len(stages)}')
for stage_name in stages:
    print('-', stage_name)

Stage count: 19
- notebook_seaborn
- notebook_yellowbrick
- notebook_detector
- notebook_art_attacks
- notebook_art_defenses
- notebook_sklearn
- notebook_pytorch
- notebook_huggingface
- notebook_hydra
- notebook_optimize
- notebook_deckard
- notebook_scoring
- notebook_dvc
- notebook_dvclive
- notebook_optuna
- notebook_lifelines
- notebook_artifacts
- notebook_fairlearn
- notebook_anjana


## Stage Shape Inspection

The next cell summarizes each stage's keys so it is easy to audit what is tracked.

This is particularly useful when deciding whether a notebook should persist `outs`, `metrics`, or `plots`.

In [2]:
from collections import Counter

summary_rows = []
for stage_name, stage_cfg in stages.items():
    keys = sorted(stage_cfg.keys()) if isinstance(stage_cfg, dict) else []
    summary_rows.append({
        'stage': stage_name,
        'has_cmd': 'cmd' in keys,
        'deps': len(stage_cfg.get('deps', [])),
        'outs': len(stage_cfg.get('outs', [])),
        'metrics': len(stage_cfg.get('metrics', [])),
        'plots': len(stage_cfg.get('plots', [])),
        'keys': keys,
    })

for row in summary_rows:
    print(row)

tracked_counter = Counter()
for row in summary_rows:
    tracked_counter['outs'] += int(row['outs'] > 0)
    tracked_counter['metrics'] += int(row['metrics'] > 0)
    tracked_counter['plots'] += int(row['plots'] > 0)

print('--- stages with tracked payloads ---')
print(dict(tracked_counter))

{'stage': 'notebook_seaborn', 'has_cmd': True, 'deps': 3, 'outs': 0, 'metrics': 0, 'plots': 6, 'keys': ['cmd', 'deps', 'plots']}
{'stage': 'notebook_yellowbrick', 'has_cmd': True, 'deps': 4, 'outs': 33, 'metrics': 0, 'plots': 0, 'keys': ['cmd', 'deps', 'outs']}
{'stage': 'notebook_detector', 'has_cmd': True, 'deps': 6, 'outs': 0, 'metrics': 1, 'plots': 0, 'keys': ['cmd', 'deps', 'metrics']}
{'stage': 'notebook_art_attacks', 'has_cmd': True, 'deps': 6, 'outs': 11, 'metrics': 3, 'plots': 1, 'keys': ['cmd', 'deps', 'metrics', 'outs', 'plots']}
{'stage': 'notebook_art_defenses', 'has_cmd': True, 'deps': 6, 'outs': 0, 'metrics': 4, 'plots': 0, 'keys': ['cmd', 'deps', 'metrics']}
{'stage': 'notebook_sklearn', 'has_cmd': True, 'deps': 10, 'outs': 2, 'metrics': 2, 'plots': 3, 'keys': ['cmd', 'deps', 'metrics', 'outs', 'plots']}
{'stage': 'notebook_pytorch', 'has_cmd': True, 'deps': 9, 'outs': 3, 'metrics': 1, 'plots': 3, 'keys': ['cmd', 'deps', 'metrics', 'outs', 'plots']}
{'stage': 'notebook_

## Focus: notebook_dvc

`notebook_dvc` is intentionally a pipeline-inspection stage in this repository.

It depends on notebook/config/code inputs but does not emit `outs`, `metrics`, or `plots`.

Runtime artifact examples are handled by sibling stages such as `notebook_dvclive` (if enabled), `notebook_optuna`, and `notebook_lifelines`.

In [3]:
stage = stages.get('notebook_dvc', {})
print('cmd:', stage.get('cmd'))
print('deps:', len(stage.get('deps', [])))
print('outs:', stage.get('outs', []))
print('metrics:', stage.get('metrics', []))
print('plots:', stage.get('plots', []))

cmd: jupyter nbconvert --to notebook --execute --inplace dvc.ipynb
deps: 10
outs: []
metrics: []
plots: []


## How To Work With This File

Common operations:
- Run one stage: `dvc repro notebook_dvc`
- Run all notebook stages: `dvc repro notebook_*`
- Force full rerun: `dvc repro --force notebook_*`

When you modify notebook dependencies or outputs, update `docs/notebooks/dvc.yaml` so CI and local execution stay in sync.